# Simulating Roland Garros 2026 with a Probabilistic Model

> *Building a Monte Carlo tennis tournament predictor in Python — inspired by probabilistic football season simulators*

---

## Motivation

In a [previous post](../atp_predictor.ipynb) I built an XGBoost classifier that predicts the outcome of any ATP match with ~66% accuracy by combining Elo ratings, clay-specific win rates, recent form, and head-to-head history.

That model answers *"who wins this match?"*.  Here I want to answer a richer question: **"who wins the whole tournament, and what is their probability of doing so?"**

The approach is borrowed directly from probabilistic football models (e.g. [this Premier League season simulator](https://medium.com/@vickyfrissdekereki/building-a-probabilistic-premier-league-simulator-in-python-34b5248f81b9)).  The key idea:

> Instead of predicting *one* deterministic outcome, we run the tournament **20 000 times** and count how often each player wins.  The resulting frequency is our probability estimate.

### Why is tennis naturally probabilistic?

Even the best player in the world loses about one-third of their matches on any given day.  A Grand Slam is seven best-of-five matches played over two weeks.  The compounding of uncertainty across seven rounds means that even a 60 % per-match win rate only translates to ~28 % tournament win probability (0.60⁷ ≈ 0.28).  Acknowledging this uncertainty explicitly — rather than pretending the model's top pick is a certainty — is the honest and useful approach.

### What makes Roland Garros special

Clay is the most surface-specific discipline in tennis.  Players like Rafael Nadal dominated clay while being merely mortal on other surfaces.  A model that ignores surface specialisation will massively misjudge a tournament played entirely on red clay.  The core of our player-strength model is therefore **clay-specific Elo**, supplemented by recent clay win rate and overall form.

---
## 1. Setup and Data Loading

We reuse the data pipeline and trained model from the ATP Predictor project.  All we need at the tournament level is:
- The **player state tracker**: an object that replays 25 years of ATP matches and tracks each player's current Elo, clay Elo, form, and H2H records.
- The **trained XGBoost model**: our match probability engine.

### Data source

This notebook uses **JeffSackmann's tennis_atp** (2000–2024) — the most reliable and widely-used historical ATP dataset. Player states (Elo, form, clay win rates) reflect match outcomes through end of 2024.

The model works equally well with any player ranking data — the article demonstrates the methodology which can be applied to more current 2026 rankings once the real Roland Garros draw is announced in late May.

In [39]:
import sys, os

# Add atp-predictor to sys.path for imports
notebook_dir = os.getcwd()
if 'roland_garros_2026' in notebook_dir:
    atp_dir = os.path.dirname(notebook_dir)
    if atp_dir not in sys.path:
        sys.path.insert(0, atp_dir)
    os.chdir(atp_dir)
else:
    atp_dir = os.path.join(os.getcwd(), 'atp-predictor')
    if os.path.exists(atp_dir):
        if atp_dir not in sys.path:
            sys.path.insert(0, atp_dir)
        os.chdir(atp_dir)

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 130

print('Working directory:', os.getcwd())
print('src importable:', os.path.exists(os.path.join(os.getcwd(), 'src')))

Working directory: c:\Users\ALESSANDRO\Documents\GitHub\tennis_analysis\atp-predictor
src importable: True


In [40]:
# ── Data pipeline ──────────────────────────────────────────────────────────
from src.data.fetch   import fetch_matches, fetch_players
from src.data.process import clean_matches

# Fetch historical data (2000–2024) from JeffSackmann's reliable repository
print("Loading ATP match data (2000-2024)...")
raw = fetch_matches(start_year=2000, end_year=2024)

# Clean and standardize matches (handles date parsing, surfaces, ranks, etc.)
matches = clean_matches(raw)

print(f"{len(matches):,} completed matches loaded")
print(f"Date range: {matches['tourney_date'].min().date()} to {matches['tourney_date'].max().date()}")

print("\n📌 Note: This uses JeffSackmann's data through December 2024.")
print("   For 2025-2026 data, the model will still work but player states")
print("   will reflect form through end of 2024. Once real RG 2026 draw is")
print("   announced, you can update the seeded_ids to reflect current rankings.")

Loading ATP match data (2000-2024)...


Loaded 74,906 matches (2000–2024)
74,906 completed matches loaded
Date range: 2000-01-03 to 2024-12-18

📌 Note: This uses JeffSackmann's data through December 2024.
   For 2025-2026 data, the model will still work but player states
   will reflect form through end of 2024. Once real RG 2026 draw is
   announced, you can update the seeded_ids to reflect current rankings.


In [41]:
# ── Build live player states ────────────────────────────────────────────────
from src.models.predict import PlayerStateTracker, MatchPredictor

tracker = PlayerStateTracker(form_window=20, surface_window_days=365)
tracker.process_all(matches)

print(f"✓ Tracking {len(tracker.name):,} players")
print(f"State current as of: {tracker._last_date.date()}")

Building state: 100%|██████████| 74906/74906 [00:04<00:00, 16146.77it/s]

✓ Tracking 2,640 players
State current as of: 2024-12-18


In [ ]:
# ── Load trained XGBoost model ──────────────────────────────────────────────
predictor = MatchPredictor(model_path='models/xgb_atp.joblib')
print('Model loaded.')

---
## 2. The Match Probability Engine

Before simulating the tournament, let's understand how match probabilities are computed.

### From Bradley-Terry to XGBoost

The simplest possible model is the **Bradley-Terry model**: assign each player a strength parameter $\lambda_i$ and predict

$$P(i \text{ beats } j) = \frac{\lambda_i}{\lambda_i + \lambda_j}$$

The **Elo rating system** is exactly Bradley-Terry with $\lambda_i = 10^{\text{Elo}_i / 400}$, which gives the familiar formula:

$$P(A \text{ beats } B) = \frac{1}{1 + 10^{(\text{Elo}_B - \text{Elo}_A)/400}}$$

This is the football model's analogue: a single strength parameter per team/player.  The Poisson goal model in football extends this to *two* parameters (attack and defence); we extend it to eleven features:

| Feature | Signal captured |
|---|---|
| `elo_diff` | Overall career quality gap |
| `surface_elo_diff` | **Clay-specific** career track record |
| `rank_diff` | Current ATP ranking gap |
| `rank_points_diff` | Points gap |
| `surf_winrate_diff` | Clay win rate, rolling 12 months |
| `form_diff` | Last 20 matches (all surfaces) |
| `p1_h2h_winrate` | Historical H2H record |
| `h2h_total` | H2H sample size |
| `age_diff` | Age gap |
| `p1_surf_n`, `p2_surf_n` | Sample size of surface data |

An XGBoost classifier learns the non-linear interactions between these features from 130 000+ historical matches.

In [ ]:
# Example: single match prediction
sinner_id  = tracker.find_player_id('Jannik Sinner')
alcaraz_id = tracker.find_player_id('Carlos Alcaraz')

if sinner_id and alcaraz_id:
    prob, info = predictor.predict_from_tracker(tracker, sinner_id, alcaraz_id, 'Clay')
    print(f"Sinner vs Alcaraz on Clay")
    print(f"  P(Sinner wins) = {prob:.1%}")
    print(f"  P(Alcaraz wins) = {1-prob:.1%}")
    print(f"  H2H: {info['p1']['name']} {info['h2h_winrate']:.0%} win rate ({info['h2h_total']} matches)")

---
## 3. Clay Court Player Profiles

For the tournament simulation we need to rank players by their **clay court ability** — this determines seedings, and seedings determine draw placement.

We compute a composite **Clay Strength Score** that mirrors the attack/defence parameters in Poisson football models.  It combines three signals:

$$\text{ClayStrength}_i = 0.45 \cdot z(\text{ClayElo}) + 0.30 \cdot z(\text{ClayWR}_{12m}) + 0.25 \cdot z(\text{Form})$$

Each component is **z-score normalised** relative to the field of 128 players, so the score measures how many standard deviations above average each dimension is.  The weights reflect our prior belief that long-run clay Elo is the most reliable predictor, followed by recent clay results, and then overall form.

This composite is used only for **seeding order**.  The actual match-by-match probabilities still come from the full XGBoost model.

In [ ]:
from roland_garros_2026.clay_rating import compute_clay_ratings

# Select top-200 ranked players who have enough data
ranked_players = [
    pid for pid, rank in tracker.rank.items()
    if rank <= 200 and pid in tracker.name
]
ranked_players.sort(key=lambda pid: tracker.rank.get(pid, 9999))

# Compute clay ratings for all of them
ratings = compute_clay_ratings(tracker, ranked_players)
print(f"Rated {len(ratings)} players")
ratings.head(16)[['name','rank','clay_elo','clay_winrate','form','clay_strength']]

In [ ]:
from roland_garros_2026.visualize import plot_clay_profile

# Load the simulation results we'll compute later — for now just plot the profile
# We'll re-run this after the simulation with actual win probabilities
# As a placeholder, add a dummy 'W' column proportional to clay_strength
_tmp = ratings.copy()
_tmp['player_id'] = _tmp['player_id']
_tmp['W'] = (_tmp['clay_strength'] - _tmp['clay_strength'].min() + 0.01)
_tmp['W'] /= _tmp['W'].sum()

fig, ax = plot_clay_profile(_tmp.rename(columns={'player_id':'player_id'}), ratings, top_n=20,
                             title='Clay Court Profile — Top 20 Contenders (pre-simulation)')
plt.show()

---
## 4. The Roland Garros Draw

Grand Slam draws follow strict seeding rules designed to protect the best players from meeting each other too early:

- **Seed 1** is placed at position 1 (top of top half).  
- **Seed 2** is placed at position 128 (bottom of bottom half).  They can only meet in the final.
- **Seeds 3–4** are drawn randomly into the two half-boundary positions.  They can only meet seeds 1 or 2 in the semis.
- **Seeds 5–8** are drawn into the four quarter-boundary positions — they enter the QFs as potential obstacles.
- **Seeds 9–16** are drawn into the eight eighth-boundary slots (potential last-16 opponents for seeds 1–8).
- **Seeds 17–32** and **unseeded players** fill remaining slots randomly.

This structure means there is genuine **draw luck**: even two players with identical underlying ability can have substantially different win probabilities depending on where the lottery places them in the bracket.

In [ ]:
from roland_garros_2026.draw import create_rg_draw, draw_to_dataframe

# Take top-128 clay-rated players as the draw field
top128_ids = ratings.head(128)['player_id'].tolist()

# Create a seeded draw (rng_seed=42 for reproducibility; change to get a different lottery)
DRAW_SEED = 42
draw = create_rg_draw(top128_ids, rng_seed=DRAW_SEED)

draw_df = draw_to_dataframe(draw, ratings)

print("First round matchups — top half (first 16 positions):")
display(draw_df.head(16)[['position','name','seed','clay_elo','clay_strength']])

In [ ]:
# Show who's in each quarter
quarters = {
    'Q1 (S1 quarter)' : draw_df[draw_df['position'].between(1,  32)],
    'Q2 (S3/4 quarter)': draw_df[draw_df['position'].between(33, 64)],
    'Q3 (S3/4 quarter)': draw_df[draw_df['position'].between(65, 96)],
    'Q4 (S2 quarter)' : draw_df[draw_df['position'].between(97, 128)],
}
for label, q in quarters.items():
    seeds_in_q = q[q['seed'].notna()]['name'].tolist()
    print(f"{label}: {', '.join(seeds_in_q[:4])}{'...' if len(seeds_in_q)>4 else ''}")

---
## 5. Monte Carlo Tournament Simulation

The simulation engine is straightforward but powerful:

```
for each of 20 000 simulations:
    copy the draw (128 players)
    for each round (R1 through Final):
        for each pair of players in the bracket:
            compute P(p1 beats p2) using XGBoost model
            randomly sample winner from Bernoulli(p)
        advance winners to next round
    record which round each player reached

for each player:
    P(reaching round R) = count(reached R) / 20 000
```

This is conceptually identical to the Monte Carlo Premier League simulator — there, each match outcome is sampled from a Poisson goal distribution; here, each match outcome is sampled from a Bernoulli distribution with probability from our XGBoost model.

**Performance note**: match probabilities are cached after the first computation, so each unique pair is only evaluated once per simulation run, not once per simulation × round.

In [ ]:
from roland_garros_2026.simulator import RGSimulator

simulator = RGSimulator(predictor=predictor, tracker=tracker)

# Run 20 000 simulations
results = simulator.simulate(draw, n_sim=20_000, seed=42)

print("\nTop 10 by championship probability:")
display(
    results.head(10)[['name','rank','clay_elo','clay_winrate','form','QF','SF','F','W']]
    .style.format({'clay_winrate':'{:.1%}','form':'{:.1%}','QF':'{:.1%}','SF':'{:.1%}','F':'{:.1%}','W':'{:.1%}'})
)

---
## 6. Results: The Round Probability Matrix

The key output is a **players × rounds probability matrix** — the same structure as the Premier League position probability heatmap, adapted for single-elimination tournaments.

Reading the heatmap:
- Each row is a player; each column is a tournament round.
- The colour and number show the probability of reaching (or surpassing) that round.
- The rapid decay from left to right reveals the compounding uncertainty across seven rounds.

This is the chart that would accompany the article on your website.

In [ ]:
from roland_garros_2026.visualize import plot_round_heatmap

fig, ax = plot_round_heatmap(results, top_n=24, title='Roland Garros 2026 — Round Probability Matrix')
plt.savefig('roland_garros_2026/rg2026_heatmap.png', bbox_inches='tight', dpi=150)
plt.show()

In [ ]:
from roland_garros_2026.visualize import plot_champion_odds

fig, ax = plot_champion_odds(results, top_n=16, title='Roland Garros 2026 — Championship Probability')
plt.savefig('roland_garros_2026/rg2026_champion_odds.png', bbox_inches='tight', dpi=150)
plt.show()

In [ ]:
from roland_garros_2026.visualize import plot_clay_profile

fig, ax = plot_clay_profile(results, ratings, top_n=20)
plt.savefig('roland_garros_2026/rg2026_clay_profile.png', bbox_inches='tight', dpi=150)
plt.show()

---
## 7. Key Matchup Preview

Who has the edge in the likely semi-finals and final?  The matchup matrix answers this by computing pairwise clay match probabilities for the top contenders.

This table is directly useful for the article — it lets readers understand *why* certain players are favoured beyond just citing a percentage.

In [ ]:
from roland_garros_2026.visualize import plot_matchup_matrix

# Top 8 by champion probability
top8_ids = results.head(8)['player_id'].tolist()

matchup_df = simulator.matchup_analysis(top8_ids)
print("Head-to-head win probabilities on clay (row beats column):")
display(matchup_df.style.background_gradient(cmap='RdYlGn', vmin=0, vmax=1).format('{:.1%}'))

In [ ]:
fig, ax = plot_matchup_matrix(matchup_df, title='Top 8 Contenders — H2H Clay Probabilities')
plt.savefig('roland_garros_2026/rg2026_matchup_matrix.png', bbox_inches='tight', dpi=150)
plt.show()

---
## 8. Draw Sensitivity: How Much Does the Lottery Matter?

One thing the Premier League simulator takes for granted is a *fixed* fixture list.  In tennis, the draw is a random lottery — two players with identical clay ability can end up in very different parts of the bracket and therefore face very different roads to the title.

We can quantify this **draw luck** by running the full simulation across many different random bracket realisations and measuring how much a player's championship probability varies.

- **High variance** → a player's fate strongly depends on the draw lottery (e.g. a mid-seed who could face the top seed in the QF or the weakest seed, depending on luck)
- **Low variance** → a player's probability is stable regardless of the draw (typically the very top players who are strong enough to beat anyone anyway)

In [ ]:
# This cell takes a few minutes — reduce n_draws for a quick test
sensitivity = simulator.draw_sensitivity(
    player_ids=top128_ids,
    n_draws=50,
    n_sim_per_draw=2_000,
    rng_seed=0,
)
print("Players most sensitive to draw luck (highest coefficient of variation):")
display(sensitivity.nlargest(10, 'win_prob_cv')[['name','win_prob_mean','win_prob_std','win_prob_cv']]
        .style.format({'win_prob_mean':'{:.1%}','win_prob_std':'{:.1%}','win_prob_cv':'{:.2f}'}))

In [ ]:
from roland_garros_2026.visualize import plot_draw_sensitivity

fig, ax = plot_draw_sensitivity(sensitivity, top_n=10)
plt.savefig('roland_garros_2026/rg2026_draw_sensitivity.png', bbox_inches='tight', dpi=150)
plt.show()

---
## 9. Summary & Article Narrative

Here is a concise summary of what the model does and what the numbers mean — ready to use as an article section.

---

### What the model does

1. **Player strength**: We assign every ATP player a *Clay Strength Score* — a weighted average of their clay Elo rating (45%), clay win rate over the last 12 months (30%), and overall recent form (25%).  This score is z-normalised relative to the draw field.

2. **Match probabilities**: For each potential matchup, we use an XGBoost classifier trained on 130 000+ historical ATP matches to estimate $P(\text{player A beats player B on clay})$.  This goes beyond simple Elo by incorporating surface win rates, form, head-to-head history, ranking points, and age.

3. **Tournament simulation**: We simulate the full 128-player bracket 20 000 times.  In each simulation, every match is sampled independently from its model probability.  Aggregating across 20 000 runs gives stable round-probability estimates.

4. **Draw sensitivity**: We repeat the simulation across 50+ different random draw realisations to separate genuine quality differences from bracket luck.

### How to read the outputs

- **Round probability matrix**: The value at row *Sinner*, column *SF* is the fraction of the 20 000 simulations in which Sinner reached the semi-finals.
- **Champion odds**: A player shown at 25% is expected to win the title in roughly one-in-four tournament realisations under this model.
- **H2H matrix**: The value at [Sinner, Alcaraz] = 0.47 means: in any one clay-court match between these two, the model gives Sinner a 47% chance.
- **Draw sensitivity**: A large error bar means the player's odds swing significantly with different bracket placements — worth noting when the real draw is announced.

### Limitations

- The training data runs through 2025; players who peak in 2026 may be underrated.
- The model is calibrated on completed matches; withdrawals and walk-overs are not modelled.
- In-tournament fatigue and injury are ignored.
- The simulation assumes match outcomes are independent — in reality, momentum effects exist.

Despite these limitations, the model produces well-calibrated match probabilities (Brier score 0.211 on 2022–2024 data) and provides a principled, transparent basis for tournament predictions.

In [ ]:
# Final summary table — copy this into your article
summary = results.head(16)[['name','rank','clay_elo','clay_winrate','form','QF','SF','F','W']].copy()
summary.index = range(1, len(summary)+1)
summary.index.name = 'seed'
summary.columns = ['Player','Rank','Clay Elo','Clay WR','Form','QF%','SF%','Final%','Win%']

display(
    summary.style
    .format({'Clay WR':'{:.1%}','Form':'{:.1%}','QF%':'{:.1%}','SF%':'{:.1%}','Final%':'{:.1%}','Win%':'{:.1%}'})
    .bar(subset=['Win%'], color='#C8421C', vmin=0)
)